In [ ]:
# %pip install kagglehub

In [ ]:
import kagglehub

path = kagglehub.dataset_download(
    "patrickfleith/nasa-anomaly-detection-dataset-smap-msl"
)



In [ ]:
from pathlib import Path

dataset_dir = Path(path)


Once you load your data, use .head() for quick inspection 

In [ ]:
import pandas as pd

# Quick inspect
df = pd.read_csv(dataset_dir / "labeled_anomalies.csv")
df.head()

At this point a data scientist would try to identify what each row and column means. What is the table size? Is anything missing or duplicated?

In [ ]:
df.shape

In [ ]:
df.info()

This csv file has stored anomaly_sequences as strings

Examine the completeness and channel identifiers with .isna().sum()

In [ ]:
df.isna().sum()

df.isna() checks every cell, returning True if it is missing and False otherwise.

.sum() adds those results down each column. True counts as 1, and False counts as 0.

# Next check to see if any channel identifiers appear more than once

In [ ]:
df['chan_id'].duplicated().sum()

This result means there is one repeated occurance of a channnel identifier 

Inspect both the original and repeated rows

In [ ]:
df.loc[df['chan_id'].duplicated(keep=False)]

# keep = False marks every occurance of a repeated identifier

P-2 appears twice as a chain_id for the same spacecraft, SMAP with different but overlaping anomaly intervals

One at row index 17

One at row index 51


.value_counts() counts how many rows contain each spacecraft name

In [ ]:
df['spacecraft'].value_counts()

In [ ]:
# Examine how much test data each channel contains
df['num_values'].describe()

.describe() retruns percentiles which describe where values fall after sorting num_values from smallest to largest 

So for example approximately 2997.5 of the total records have this many test readings or fewer

count is the number of non-missing entires in the column. So here, count is 82 which means there are 82 rows with a num_values value. It does not mean 82 sensor readings, it tells us how many test redings belong to that rows channel

In [ ]:
import matplotlib.pyplot as plt

df['num_values'].hist(bins= 20)
plt.xlabel('Number of test readings')
plt.ylabel('Number of channel records') #channel meaning the stream of readings for a monitored quantity
# this data set has channel names P-1, S-1, E-1 etc

plt.title("Test sequence lengths")
plt.show()

Do most channels have similar amounts of data? Do lengths vary substantially? 

Yes it seems like there is a large concentration of readings near 8-8.5k

# Seaborn

Recall its a plotting library that works well with pandas dataframes 

you provide the data and column names and it handels much of the plots appearance

A few useful things to consider

1. sns.countplot() 
This tells us how mnay channel records belong to each spacecraft

2. sns.boxplot()
This tells us how sequence lengths differ between spacecraft

3. sns.lineplot()
This tells us how a telemetry reading changes overtime once we load a channel

4. sns.scatterplot()
This tells us if two numerical variables have a relationship

5. sns.pairplot()
This helps us explore several numerical columns at once. This data only has 1 numerical column so it might not be as helpful in this experience

In [ ]:
import seaborn as sns

sns.boxplot(data = df, x = 'spacecraft', y = 'num_values')
plt.xlabel("Spacecraft")
plt.ylabel("Number of Readings")
plt.title("Test Sequence Length's by Spacecraft")

plt.show()

The line inside each box is the median value, and the box spans the 25th through 75th percentiles

Points beyond the whiskers indicate unusually short or long sequences within that group

# lets try and plot more specific data channels

To see what we are working with, first figure out what all the data channels are


In [ ]:
sorted(df['chan_id'].unique())

These are all the data channels, lets sort by spacecraft

In [ ]:
with pd.option_context('display.max_rows', None):
    display(df[['chan_id','spacecraft','class', 'anomaly_sequences']].sort_values('chan_id'))


# df[['chan_id','spacecraft']].drop_duplicates()

### SOMEHTING IMPORTANT TO POINT OUT

All these data channels are already split into training data set and a test data set for us. Downloading the data automatically downloads all the folders in which you have a training set and a test set

### The downloaded labeled_anomalies.csv file describes where known anomalies occur within each channels test data

But this does not necessarily mean that we shouldn't try to predict where anomolies will occur. Because we know where they occur, we can use them to check the accuracy of our predictive models



### Exploratory use of test data

Normaly we'd wait until we finish training the model before applying it to test data. 

In the following examples I inspected the supplied the actual test readings to make predictions when selecting the method I wanted to use

Included are also anomaly labels to understand the dataset and understand how Isolation Forest works. 

Again: the models are fitted only on training data, but test readings are used to generate predictions which show up as orange dots marking model-flagged readings in the following plots. 


These results are merely for example, not final evaluation since test data was used to make predctions

# Example using P-1 
[contextual, contextual, contextual]

For now, here is a plot of the raw data

In [ ]:
import numpy as np
import ast

# Load P-1's test readings
p1_test = np.load(dataset_dir / "data" / "data" / "test" / "P-1.npy")

# Get its anomaly intervals
intervals = ast.literal_eval(
    df.loc[df['chan_id'] == 'P-1', 'anomaly_sequences'].iloc[0]
)

# so from ast we get: intervals = [[2149, 2349], [4536, 4844], [3539, 3779]]

fig, ax = plt.subplots(figsize=(12, 4))

sns.lineplot(
    x=np.arange(len(p1_test)),
    y=p1_test[:, 0],
    ax=ax
)

for i, (start, end) in enumerate(intervals):
    ax.axvspan(
        start, end,
        color="red",
        alpha=0.2,
        label="Labeled anomaly" if i == 0 else None
    )
    
# this loop takes a pair in the interval seq, unpacks them each time it loops
# so for example: pass 1 start = 2149, end = 2349

ax.set_xlabel("Time-step index in test data")
ax.set_ylabel("Scaled telemetry value")
ax.set_title("P-1 test telemetry with labeled anomalies")
# ax.set_xlim(1900, 2600)
ax.legend()
plt.show()

# Note
Scaled telemetry value means the readings have been transformed into a numerical scale rather than shown in their original units, such as volts or degrees

[https://github.com/khundman/telemanom]

The dataset creators scaled the telemetry using each channles test-set minimum and maximum mapping them to -1 and +1

-1 represents the lowest recorded value

+1 represents the highest recorded value

0 is a midpoint betweeen those extremes

### (so this shows us how high or low a reading is, not how unusual it is. An anomaly socre would be a separate quantitiy calculated by a detection algorithim for example)

In [ ]:
# Zooming in to one anomoly region

fig, ax = plt.subplots(figsize=(12, 4))

sns.lineplot(
    x=np.arange(len(p1_test)),
    y=p1_test[:, 0],
    ax=ax
)

for i, (start, end) in enumerate(intervals):
    ax.axvspan(
        start, end,
        color="red",
        alpha=0.2,
        label="Labeled anomaly" if i == 0 else None
    )

ax.set_xlabel("Time-step index in test data")
ax.set_ylabel("Scaled telemetry value")
ax.set_title("P-1 test telemetry with labeled anomalies")
ax.set_xlim(1900, 2600)
ax.legend()
plt.show()

# Next, an example using P- 4

[point, point, point]

In [ ]:
# Load D-14's test readings
P4_test = np.load(dataset_dir / "data" / "data" / "test" / "P-4.npy")

# Get its anomaly intervals
intervals = ast.literal_eval(
    df.loc[df['chan_id'] == 'P-4', 'anomaly_sequences'].iloc[0]
)


fig, ax = plt.subplots(figsize=(12, 4))

sns.lineplot(
    x=np.arange(len(P4_test)),
    y=P4_test[:, 0],
    ax=ax
)

for i, (start, end) in enumerate(intervals):
    ax.axvspan(
        start, end,
        color="red",
        alpha=0.2,
        label="Labeled anomaly" if i == 0 else None
    )
    


ax.set_xlabel("Time-step index in test data")
ax.set_ylabel("Scaled telemetry value")
ax.set_title("P- 4 test telemetry with labeled anomalies")
# ax.set_xlim(1900, 2600)
ax.legend()
plt.show()

# One tool in Scikit learn that stood out to me was Isolation Forrest. The following cells attempt to implement this method to our data

### Isolation Forest: How does it consider when a value is unusual?

This method uses random splits to separate observations. Those isolated in relatively few splits recieve a higher anomoly rating. The key here is that the data is split randomly, as opposed to other methods of interest that I encountered (Elliptic Envelope , OneClassSVM ) where we have more control as to how we want to define "unusual" when it comes to a data value

#### What question does Isolation Forest allow us to test specifically?

Does random partitioning separate anomolies more effectively including when the data has a more complicated shape?


#### To begin I selected 1 data channel as an example: D-14

In [ ]:
from sklearn.ensemble import IsolationForest

# Load the supplied training data
D14_train = np.load(
    dataset_dir / "data" / "data" / "train" / "D-14.npy"
)

D14_test = np.load(
    dataset_dir / "data" / "data" / "test" / "D-14.npy" )

# Keep the telemetry column as a two-dimensional array
X_train = D14_train[:, [0]]
X_test = D14_test[:, [0]]

model = IsolationForest(random_state=42)

# Learn from training readings without anomaly labels
model.fit(X_train)

# Flag unusual test readings
predictions = model.predict(X_test)

In [ ]:
print("Total test readings:", len(predictions))
print(f"Percentage flagged: {(predictions == -1).mean():.1%}")

### Immediate concern: ALL the data values in the D-14 data channel were flagged as unusual by this algorithim. Why?

In [ ]:
for name, values in [
    ("Train", D14_train[:, 0]),
    ("Test", D14_test[:, 0])
]:
    print(name)
    print(pd.Series(values).describe())

In [ ]:
sns.histplot(D14_train[:, 0], color="blue",
             label="Train", stat="density", alpha=0.4)
sns.histplot(D14_test[:, 0], color="orange",
             label="Test", stat="density", alpha=0.4)

plt.xlabel("Scaled telemetry value")
plt.ylabel("Density")
plt.title("D-14 training and test distributions")
plt.legend()
plt.show()

This histogram compares D-14’s telemetry values in the training and test sets.

- X-axis: scaled telemetry value.
- Blue bars: training distribution all readings equal −1.
- Orange bars: test distribution most readings equal −1, but some reach higher values, near +1.
- Y-axis (density): normalized frequency. each distribution’s total bar area equals 1.


D-14’s training telemetry is constant, while its test telemetry includes departures from that value. With only this constant training feature, Isolation Forest cannot build useful splits

#### So the reason the algorithim failed could be because there is no standard deviation between the training values.

Every single training reading is exactly -1, while most of the test readings are also -1 

# This exposes a limitation of the Isolation Forrest Method and useful finding

It needs differences between training observations to make its splits. With just one constant telemetry value, it cannot build useful partitions so this method may not be a good one to use for data channels with [point] anomolies

#### The following cells conduct another implementation of Isolation Forrest with the P-1 data channel

In [ ]:
p1_train = np.load(
    dataset_dir / "data" / "data" / "train" / "P-1.npy"
)

print(pd.Series(p1_train[:, 0]).describe())

In [ ]:
p1_test = np.load(
    dataset_dir / "data" / "data" / "test" / "P-1.npy"
)

X_train = p1_train[:, [0]]
X_test = p1_test[:, [0]]

p1_model = IsolationForest(random_state=42)
p1_model.fit(X_train)

p1_predictions = p1_model.predict(X_test)

print("Total test readings:", len(p1_predictions))
print("Flagged readings:", (p1_predictions == -1).sum())
print(f"Percentage flagged: {(p1_predictions == -1).mean():.1%}")

In [ ]:
intervals = ast.literal_eval(
    df.loc[df['chan_id'] == 'P-1', 'anomaly_sequences'].iloc[0]
)

flagged = p1_predictions == -1
time_steps = np.arange(len(p1_test))

fig, ax = plt.subplots(figsize=(12, 4))

sns.lineplot(x=time_steps, y=p1_test[:, 0], ax=ax)

ax.scatter(
    time_steps[flagged],
    p1_test[flagged, 0],
    color="orange",
    s=12,
    label="Model flag",
    zorder=3
)

for i, (start, end) in enumerate(intervals):
    ax.axvspan(
        start, end,
        color="red",
        alpha=0.2,
        label="Known anomaly interval" if i == 0 else None
    )

ax.set_xlabel("Time-step index in test data")
ax.set_ylabel("Scaled telemetry value")
ax.set_title("P-1: Isolation Forest flags and known anomalies")
ax.set_xlim(1900, 2600)

ax.legend()
plt.show()

The model flags values both inside and out of the known anomoly interval

But it seems like the flat stretch at around -8 between the time steps (2190 - 2260) recieves no flags. I think this is what should have gotten a flag


#### Again the fitted model evaluated the test readings to generate anomaly flags. Again, this is not the proper way to do this, but I did it merely for example

In [ ]:
# Zooming into another anomoly interval

intervals = ast.literal_eval(
    df.loc[df['chan_id'] == 'P-1', 'anomaly_sequences'].iloc[0]
)

flagged = p1_predictions == -1
time_steps = np.arange(len(p1_test))

fig, ax = plt.subplots(figsize=(12, 4))

sns.lineplot(x=time_steps, y=p1_test[:, 0], ax=ax)

ax.scatter(
    time_steps[flagged],
    p1_test[flagged, 0],
    color="orange",
    s=12,
    label="Model flag",
    zorder=3
)

for i, (start, end) in enumerate(intervals):
    ax.axvspan(
        start, end,
        color="red",
        alpha=0.2,
        label="Known anomaly interval" if i == 0 else None
    )

ax.set_xlabel("Time-step index in test data")
ax.set_ylabel("Scaled telemetry value")
ax.set_title("P-1: Isolation Forest flags and known anomalies")
ax.set_xlim(3400, 3900)

ax.legend()
plt.show()

### Some labeled anomalous readings recieve no flags

With only the current reading as the input, the model cannot examine preceding readings or how long a value persits

Unlike supervised models fitted with both X and known y targets, Isolation Forest learns from X alone. After fitting, it produces flags for new observations 


#### What would happen if we gave the Isolation Forrest some historial context of the data set?

Meaning, what happens if I supply a current reading in the time series as these plots have done, a previous reading, and one from two steps ago?


For the following demonstration I supply ONLY the training data and give the model validation readings instead of test readings for it to raise flags


I use an 80 -20 split on the supplied trainning file

The model learns from 80 percent of the data using model.fit(X_fit)

The rest of the 20 percent in the training file is now used in place of the testing set to predict flags using model.predict(X_val)


The following 4 cells should be run AFTER restarting and clearing all outputs. After restarting and clearing, do not run any of the previous cells only the last 4. The model algorithim will perform much better.

In [ ]:
# def make_history_features(data):
#     readings = pd.Series(data[:, 0])

#     features = pd.DataFrame({
#         "current": readings,
#         "previous_1": readings.shift(1),
#         "previous_2": readings.shift(2)
#     })

#     return features.dropna()

# X_train_history = make_history_features(p1_train)
# X_test_history = make_history_features(p1_test)

# X_train_history.head()

# p1_history_model = IsolationForest(random_state=42)
# p1_history_model.fit(X_train_history)

# history_predictions = p1_history_model.predict(X_test_history)

# print("Evaluated readings:", len(history_predictions))
# print(f"Percentage flagged: {(history_predictions == -1).mean():.1%}")

In [ ]:
# #make plot using only training data in p-1


# from pathlib import Path
# import numpy as np
# import pandas as pd
# import seaborn as sns
# import matplotlib.pyplot as plt
# from sklearn.ensemble import IsolationForest

# dataset_dir = Path(
#     "/home/diegorios/.cache/kagglehub/datasets/"
#     "patrickfleith/nasa-anomaly-detection-dataset-smap-msl/versions/1"
# )

# # Load ONLY the supplied training file
# p1_train = np.load(dataset_dir / "data" / "data" / "train" / "P-1.npy")

# # Put the telemetry readings in a Series with their original row indices
# readings = pd.Series(p1_train[:, 0], name="current")

# # Reserve the later 20% for validation
# split = int(0.8 * len(readings))
# fit_readings = readings.iloc[:split]
# val_readings = readings.iloc[split:]

In [ ]:
# def make_history_features(readings):
#     return pd.DataFrame({
#         "current": readings,
#         "previous_1": readings.shift(1),
#         "previous_2": readings.shift(2)
#     }).dropna()

# X_fit = make_history_features(fit_readings)
# X_val = make_history_features(val_readings)

# model = IsolationForest(random_state=42)
# model.fit(X_fit)

# val_predictions = model.predict(X_val)

# print("Validation readings:", len(val_predictions))
# print(f"Percentage flagged: {(val_predictions == -1).mean():.1%}")

In [ ]:
# flagged = val_predictions == -1
# time_steps = X_val.index.to_numpy()

# fig, ax = plt.subplots(figsize=(12, 4))

# sns.lineplot(x=time_steps, y=X_val["current"], ax=ax)

# ax.scatter(
#     time_steps[flagged],
#     X_val.loc[flagged, "current"],
#     color="orange",
#     s=15,
#     label="Model flag",
#     zorder=3
# )

# ax.set_xlabel("Time-step index in supplied training data")
# ax.set_ylabel("Scaled telemetry value")
# ax.set_title("P-1: validation within the supplied training sequence")
# # ax.set_xlim(2350, 2400)

# ax.legend()
# plt.show()